In [7]:
# ===========================
# 1. IMPORTS
# ===========================
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

from tensorflow.keras import Input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense


# ===========================
# 2. LOAD DATA
# ===========================
X_images = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_images.npy")
X_rna = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_rna.npy")
X_clinical = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_clinical.npy")
y = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\y.npy")

print("Loaded Data:")
print(X_images.shape, X_rna.shape, X_clinical.shape, y.shape)


# ===========================
# 3. RNA REDUCTION (PCA)
# ===========================
pca = PCA(n_components=20)
X_rna = pca.fit_transform(X_rna)

print("RNA reduced:", X_rna.shape)


# ===========================
# 4. IMAGE FEATURE EXTRACTION
# ===========================
# reshape patients → slices
X_img = X_images.reshape(-1, 224, 224, 1)

# -------- CNN (Functional API) --------
input_layer = Input(shape=(224, 224, 1))

x = Conv2D(32, (3,3), activation='relu')(input_layer)
x = MaxPooling2D(2,2)(x)

x = Conv2D(64, (3,3), activation='relu')(x)
x = MaxPooling2D(2,2)(x)

x = Conv2D(128, (3,3), activation='relu')(x)
x = MaxPooling2D(2,2)(x)

x = Flatten()(x)
x = Dense(128, activation='relu')(x)

# feature extractor
feature_extractor = Model(inputs=input_layer, outputs=x)

# extract slice features
X_img_features = feature_extractor.predict(X_img, verbose=1)

# reshape back to patients
X_img_features = X_img_features.reshape(X_images.shape[0], 50, 128)
X_img_features = np.mean(X_img_features, axis=1)

print("Image features:", X_img_features.shape)


# ===========================
# 5. CLINICAL MODEL (DENSENET STYLE)
# ===========================
class DenseBlock(nn.Module):
    def __init__(self, input_dim, growth_rate):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(input_dim, growth_rate),
            nn.ReLU()
        )

    def forward(self, x):
        out = self.layer(x)
        return torch.cat([x, out], dim=1)


class DenseNetTabular(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.block1 = DenseBlock(input_dim, 32)
        self.block2 = DenseBlock(input_dim + 32, 32)
        self.block3 = DenseBlock(input_dim + 64, 32)

        self.final = nn.Sequential(
            nn.Linear(input_dim + 96, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.final(x)


# convert clinical to tensor
X_clinical_t = torch.tensor(X_clinical, dtype=torch.float32)

clinical_model = DenseNetTabular(X_clinical.shape[1])

with torch.no_grad():
    clinical_features = clinical_model(X_clinical_t).numpy()

print("Clinical features:", clinical_features.shape)


# ===========================
# 6. FUSION
# ===========================
X_fused = np.concatenate([
    X_img_features,
    X_rna,
    clinical_features
], axis=1)

print("Fused shape:", X_fused.shape)


# ===========================
# 7. FINAL MODEL
# ===========================
scaler = StandardScaler()
X_fused = scaler.fit_transform(X_fused)

X_train, X_test, y_train, y_test = train_test_split(
    X_fused, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# convert to tensor
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1,1)

X_test_t = torch.tensor(X_test, dtype=torch.float32)


# PyTorch classifier
model = nn.Sequential(
    nn.Linear(X_fused.shape[1], 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


# ===========================
# 8. TRAIN
# ===========================
epochs = 100

for epoch in range(epochs):
    model.train()
    
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


# ===========================
# 9. EVALUATION
# ===========================
model.eval()

with torch.no_grad():
    logits = model(X_test_t)
    preds = torch.sigmoid(logits).numpy()

preds_binary = (preds > 0.5).astype(int)

print("\n===== FINAL FUSION MODEL =====")
print("Accuracy:", accuracy_score(y_test, preds_binary))
print("F1 Score:", f1_score(y_test, preds_binary))

Loaded Data:
(42, 50, 224, 224, 1) (42, 20) (42, 76) (42,)
RNA reduced: (42, 20)
66/66 ━━━━━━━━━━━━━━━━━━━━ 8s 110ms/step
Image features: (42, 128)
Clinical features: (42, 16)
Fused shape: (42, 164)
Epoch 10, Loss: 0.6448
Epoch 20, Loss: 0.5511
Epoch 30, Loss: 0.4311
Epoch 40, Loss: 0.3120
Epoch 50, Loss: 0.2052
Epoch 60, Loss: 0.1177
Epoch 70, Loss: 0.0664
Epoch 80, Loss: 0.0393
Epoch 90, Loss: 0.0270
Epoch 100, Loss: 0.0174

===== FINAL FUSION MODEL =====
Accuracy: 0.4444444444444444
F1 Score: 0.0


In [11]:
# ===========================
# 1. IMPORTS
# ===========================
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input


# ===========================
# 2. LOAD DATA
# ===========================
X_images = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_images.npy")
X_rna = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_rna.npy")
X_clinical = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_clinical.npy")
y = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\y.npy")

print("Loaded:", X_images.shape, X_rna.shape, X_clinical.shape)


# ===========================
# 3. RNA REDUCTION
# ===========================
pca = PCA(n_components=10)
X_rna = pca.fit_transform(X_rna)


# ===========================
# 4. IMAGE FEATURE EXTRACTION (PRETRAINED)
# ===========================
n_patients, n_slices = X_images.shape[0], X_images.shape[1]

# reshape slices
X_img = X_images.reshape(-1, 224, 224, 1)

# convert grayscale → RGB
X_img = np.repeat(X_img, 3, axis=-1)

# preprocess
X_img = preprocess_input(X_img)

# load pretrained model
base_model = MobileNetV3Small(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

# extract features
features = base_model.predict(X_img, verbose=1)

# global pooling
features = np.mean(features, axis=(1,2))  # shape: (total_slices, channels)

# reshape back to patients
features = features.reshape(n_patients, n_slices, -1)

# aggregate slices
X_img_features = np.mean(features, axis=1)

# reduce dimension
pca_img = PCA(n_components=20)
X_img_features = pca_img.fit_transform(X_img_features)

print("Image features:", X_img_features.shape)


# ===========================
# 5. CLINICAL FEATURES
# ===========================
# use directly (already processed)
X_clinical_final = X_clinical


# ===========================
# 6. FUSION
# ===========================
X_fused = np.concatenate([
    X_img_features,
    X_rna,
    X_clinical_final
], axis=1)

scaler = StandardScaler()
X_fused = scaler.fit_transform(X_fused)

print("Fused shape:", X_fused.shape)


# ===========================
# 7. MODEL DEFINITION
# ===========================
class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


# ===========================
# 8. CROSS VALIDATION
# ===========================
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

acc_scores = []
f1_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X_fused, y)):
    print(f"\n===== FOLD {fold+1} =====")

    X_train, X_test = X_fused[train_idx], X_fused[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)

    model = SimpleNN(X_fused.shape[1])

    # class imbalance handling
    pos_weight = torch.tensor([(len(y_train) - sum(y_train)) / sum(y_train)])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # TRAIN
    for epoch in range(80):
        model.train()

        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # EVALUATE
    model.eval()
    with torch.no_grad():
        logits = model(X_test_t)
        preds = torch.sigmoid(logits).numpy()

    # threshold tuning
    best_f1 = 0
    best_thresh = 0.5

    for t in np.arange(0.1, 0.9, 0.05):
        temp_preds = (preds > t).astype(int)
        f1 = f1_score(y_test, temp_preds)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    final_preds = (preds > best_thresh).astype(int)

    acc = accuracy_score(y_test, final_preds)
    f1 = f1_score(y_test, final_preds)

    print(f"Best Threshold: {best_thresh:.2f}")
    print(f"Accuracy: {acc:.3f}, F1: {f1:.3f}")

    acc_scores.append(acc)
    f1_scores.append(f1)


# ===========================
# 9. FINAL RESULTS
# ===========================
print("\n===== FINAL RESULTS =====")
print("Mean Accuracy:", np.mean(acc_scores))
print("Mean F1 Score:", np.mean(f1_scores))

Loaded: (42, 50, 224, 224, 1) (42, 20) (42, 76)
66/66 ━━━━━━━━━━━━━━━━━━━━ 11s 148ms/step
Image features: (42, 20)
Fused shape: (42, 106)

===== FOLD 1 =====
Best Threshold: 0.50
Accuracy: 0.778, F1: 0.667

===== FOLD 2 =====
Best Threshold: 0.55
Accuracy: 0.889, F1: 0.857

===== FOLD 3 =====
Best Threshold: 0.10
Accuracy: 0.375, F1: 0.545

===== FOLD 4 =====
Best Threshold: 0.50
Accuracy: 0.875, F1: 0.800

===== FOLD 5 =====
Best Threshold: 0.50
Accuracy: 0.625, F1: 0.571

===== FINAL RESULTS =====
Mean Accuracy: 0.7083333333333333
Mean F1 Score: 0.6881385281385282


In [13]:
# ===========================
# 1. IMPORTS
# ===========================
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input


# ===========================
# 2. LOAD DATA
# ===========================
X_images = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_images.npy")
X_rna = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_rna.npy")
X_clinical = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_clinical.npy")
y = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\y.npy")

print("Loaded:", X_images.shape, X_rna.shape, X_clinical.shape)


# ===========================
# 3. RNA REDUCTION
# ===========================
pca = PCA(n_components=20)
X_rna = pca.fit_transform(X_rna)


# ===========================
# 4. IMAGE FEATURE EXTRACTION (PRETRAINED)
# ===========================
n_patients, n_slices = X_images.shape[0], X_images.shape[1]

# reshape slices
X_img = X_images.reshape(-1, 224, 224, 1)

# convert grayscale → RGB
X_img = np.repeat(X_img, 3, axis=-1)

# preprocess
X_img = preprocess_input(X_img)

# load pretrained model
base_model = MobileNetV3Small(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

# extract features
features = base_model.predict(X_img, verbose=1)

# global pooling
features = np.mean(features, axis=(1,2))  # shape: (total_slices, channels)

# reshape back to patients
features = features.reshape(n_patients, n_slices, -1)

# aggregate slices
X_img_features = np.mean(features, axis=1)

# reduce dimension
pca_img = PCA(n_components=20)
X_img_features = pca_img.fit_transform(X_img_features)

print("Image features:", X_img_features.shape)


# ===========================
# 5. CLINICAL FEATURES
# ===========================
# use directly (already processed)
X_clinical_final = X_clinical


# ===========================
# 6. FUSION
# ===========================
X_fused = np.concatenate([
    X_img_features,
    X_rna,
    X_clinical_final
], axis=1)

scaler = StandardScaler()
X_fused = scaler.fit_transform(X_fused)

print("Fused shape:", X_fused.shape)


# ===========================
# 7. MODEL DEFINITION
# ===========================
class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


# ===========================
# 8. CROSS VALIDATION
# ===========================
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

acc_scores = []
f1_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X_fused, y)):
    print(f"\n===== FOLD {fold+1} =====")

    X_train, X_test = X_fused[train_idx], X_fused[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)

    model = SimpleNN(X_fused.shape[1])

    # class imbalance handling
    pos_weight = torch.tensor([(len(y_train) - sum(y_train)) / sum(y_train)])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # TRAIN
    for epoch in range(80):
        model.train()

        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # EVALUATE
    model.eval()
    with torch.no_grad():
        logits = model(X_test_t)
        preds = torch.sigmoid(logits).numpy()

    # threshold tuning
    best_f1 = 0
    best_thresh = 0.5

    for t in np.arange(0.1, 0.9, 0.05):
        temp_preds = (preds > t).astype(int)
        f1 = f1_score(y_test, temp_preds)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    final_preds = (preds > best_thresh).astype(int)

    acc = accuracy_score(y_test, final_preds)
    f1 = f1_score(y_test, final_preds)

    print(f"Best Threshold: {best_thresh:.2f}")
    print(f"Accuracy: {acc:.3f}, F1: {f1:.3f}")

    acc_scores.append(acc)
    f1_scores.append(f1)


# ===========================
# 9. FINAL RESULTS
# ===========================
print("\n===== FINAL RESULTS =====")
print("Mean Accuracy:", np.mean(acc_scores))
print("Mean F1 Score:", np.mean(f1_scores))

Loaded: (42, 50, 224, 224, 1) (42, 20) (42, 76)
66/66 ━━━━━━━━━━━━━━━━━━━━ 13s 169ms/step
Image features: (42, 20)
Fused shape: (42, 116)

===== FOLD 1 =====
Best Threshold: 0.45
Accuracy: 0.889, F1: 0.800

===== FOLD 2 =====
Best Threshold: 0.55
Accuracy: 1.000, F1: 1.000

===== FOLD 3 =====
Best Threshold: 0.10
Accuracy: 0.375, F1: 0.545

===== FOLD 4 =====
Best Threshold: 0.40
Accuracy: 0.750, F1: 0.667

===== FOLD 5 =====
Best Threshold: 0.70
Accuracy: 0.875, F1: 0.800

===== FINAL RESULTS =====
Mean Accuracy: 0.7777777777777778
Mean F1 Score: 0.7624242424242424


In [19]:
# ===========================
# 1. IMPORTS
# ===========================
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input


# ===========================
# 2. LOAD DATA
# ===========================
X_images = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_images.npy")
X_rna = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_rna.npy")
X_clinical = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\X_clinical.npy")
y = np.load(r"C:\Users\hudda\OneDrive\Desktop\EIS\processed_data\y.npy")

print("Loaded:", X_images.shape, X_rna.shape, X_clinical.shape)


# ===========================
# 3. RNA REDUCTION
# ===========================
pca = PCA(n_components=20)
X_rna = pca.fit_transform(X_rna)


# ===========================
# 4. IMAGE FEATURE EXTRACTION (PRETRAINED)
# ===========================
n_patients, n_slices = X_images.shape[0], X_images.shape[1]

# reshape slices
X_img = X_images.reshape(-1, 224, 224, 1)

# convert grayscale → RGB
X_img = np.repeat(X_img, 3, axis=-1)

# preprocess
X_img = preprocess_input(X_img)

# load pretrained model
base_model = MobileNetV3Small(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

# extract features
features = base_model.predict(X_img, verbose=1)

# global pooling
features = np.mean(features, axis=(1,2))  # shape: (total_slices, channels)

# reshape back to patients
features = features.reshape(n_patients, n_slices, -1)

# aggregate slices
X_img_features = np.mean(features, axis=1)

# reduce dimension
pca_img = PCA(n_components=20)
X_img_features = pca_img.fit_transform(X_img_features)

print("Image features:", X_img_features.shape)


# ===========================
# 5. CLINICAL FEATURES
# ===========================
# use directly (already processed)
X_clinical_final = X_clinical


# ===========================
# 6. FUSION
# ===========================
X_fused = np.concatenate([
    X_img_features,
    X_rna,
    X_clinical_final
], axis=1)

scaler = StandardScaler()
X_fused = scaler.fit_transform(X_fused)

print("Fused shape:", X_fused.shape)


# ===========================
# 7. MODEL DEFINITION
# ===========================
class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


# ===========================
# 8. CROSS VALIDATION
# ===========================
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

acc_scores = []
f1_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X_fused, y)):
    print(f"\n===== FOLD {fold+1} =====")

    X_train, X_test = X_fused[train_idx], X_fused[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)

    model = SimpleNN(X_fused.shape[1])

    # class imbalance handling
    pos_weight = torch.tensor([(len(y_train) - sum(y_train)) / sum(y_train)])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # TRAIN
    for epoch in range(80):
        model.train()

        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # EVALUATE
    model.eval()
    with torch.no_grad():
        logits = model(X_test_t)
        preds = torch.sigmoid(logits).numpy()

    # threshold tuning
    best_f1 = 0
    best_thresh = 0.5

    for t in np.arange(0.1, 0.9, 0.05):
        temp_preds = (preds > t).astype(int)
        f1 = f1_score(y_test, temp_preds)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    final_preds = (preds > best_thresh).astype(int)

    acc = accuracy_score(y_test, final_preds)
    f1 = f1_score(y_test, final_preds)

    print(f"Best Threshold: {best_thresh:.2f}")
    print(f"Accuracy: {acc:.3f}, F1: {f1:.3f}")

    acc_scores.append(acc)
    f1_scores.append(f1)


# ===========================
# 9. FINAL RESULTS
# ===========================
print("\n===== FINAL RESULTS =====")
print("Mean Accuracy:", np.mean(acc_scores))
print("Mean F1 Score:", np.mean(f1_scores))

Loaded: (42, 50, 224, 224, 1) (42, 20) (42, 76)
66/66 ━━━━━━━━━━━━━━━━━━━━ 11s 148ms/step
Image features: (42, 20)
Fused shape: (42, 116)

===== FOLD 1 =====
Best Threshold: 0.50
Accuracy: 0.889, F1: 0.800

===== FOLD 2 =====
Best Threshold: 0.55
Accuracy: 1.000, F1: 1.000

===== FOLD 3 =====
Best Threshold: 0.50
Accuracy: 0.625, F1: 0.571

===== FOLD 4 =====
Best Threshold: 0.55
Accuracy: 0.875, F1: 0.667

===== FOLD 5 =====
Best Threshold: 0.55
Accuracy: 0.750, F1: 0.667

===== FINAL RESULTS =====
Mean Accuracy: 0.8277777777777778
Mean F1 Score: 0.7409523809523809
